https://dropsofai.com/sentiment-analysis-with-python-bag-of-words/

In [ ]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
import nltk
import sklearn
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook
%matplotlib inline

In [ ]:
TEXTBLOB = False
SAVE_IMAGES = False
Daily = True; Hourly = False
assert(not(Daily & Hourly))
LINEAR_REGRESSION = False

In [ ]:
def remove_html(text):
    bs = BeautifulSoup(text, "html.parser")
    return ' ' + bs.get_text() + ' '
 
def keep_only_letters(text):
    text=re.sub(r'[^a-zA-Z\s]','',text)
    return text
 
def convert_to_lowercase(text):
    return text.lower()
 
def clean_reviews(text):
    text = remove_html(text)
    text = keep_only_letters(text)
    text = convert_to_lowercase(text)
    return text

In [ ]:
english_stop_words = nltk.corpus.stopwords.words('english')
print(len(english_stop_words))
print (english_stop_words[:20])

In [ ]:
if (Daily):
    %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"
elif (Hourly):
    %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/InitializationHourly.ipynb"
    df = df.loc[(df["Hour"]==15)]# | (df_part3["Hour"]==15)]


train_start_date = '2020-05-31'
train_end_date = '2021-05-31'
test_start_date = '2021-05-31'
test_end_date = '2021-10-1'
df = df.ffill()

train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date : test_end_date]


press_release_train = []

press_release_train = train.Articles

press_release_test = []
press_release_test = test.Articles

In [ ]:
def remove_stop_words(text):
    for stopword in english_stop_words:
        stopword = ' ' + stopword + ' '
        text = text.replace(stopword, ' ')
    return text
 
df['Articles'] = df['Articles'].apply(remove_stop_words)

In [ ]:
def text_stemming(text):
    stemmer = nltk.porter.PorterStemmer()
    stemmed = ' '.join([stemmer.stem(token) for token in text.split()])
    return stemmed
 
df['Articles'] = df['Articles'].apply(text_stemming)

In [ ]:
vectorizer = sklearn.feature_extraction.text.CountVectorizer(binary=False,ngram_range=(1,1))
tf_features_train = vectorizer.fit_transform(press_release_train)
tf_features_test = vectorizer.transform(press_release_test)
print (tf_features_train.shape, tf_features_test.shape)

In [ ]:
train_labels = train.pos
test_labels = test.pos
train_labels = [1 if sentiment>0 else 0 for sentiment in train['pos']]
test_labels = [1 if sentiment>0 else 0 for sentiment in test['pos']]

print (len(train_labels), len(test_labels))

In [ ]:
clf = sklearn.linear_model.LogisticRegression()
clf.fit(tf_features_train, train_labels)
print (clf)